In [1]:
import pickle
import torch
from tqdm import tqdm

def load(filename):
    path = f"./data/cifar-10-batches-py/{filename}"
    with open(path, 'rb') as fo: return pickle.load(fo, encoding='bytes')

def load_cifar10():
    batches = [load(f"data_batch_{i}") for i in range(1, 6)]
    X_tr = torch.cat([torch.as_tensor(b[b"data"], dtype=torch.float32) for b in batches])
    Y_tr = torch.cat([torch.as_tensor(b[b"labels"], dtype=torch.long) for b in batches])

    test = load("test_batch")
    X_te = torch.as_tensor(test[b"data"], dtype=torch.float32)
    Y_te = torch.as_tensor(test[b"labels"], dtype=torch.long)

    return X_tr, Y_tr, X_te, Y_te

X_tr, Y_tr, X_te, Y_te = load_cifar10()

print(f"{X_tr.shape = }")
print(f"{Y_tr.shape = }")
print(f"{X_te.shape = }")
print(f"{Y_te.shape = }")

X_tr.shape = torch.Size([50000, 3072])
Y_tr.shape = torch.Size([50000])
X_te.shape = torch.Size([10000, 3072])
Y_te.shape = torch.Size([10000])


In [2]:
class NearestNeighbor:
    METRICS = {"l1": (torch.int16, torch.abs), "l2": (torch.int32, torch.square)}

    def __init__(self, metric = "l1"): self.dtype, self.fn = self.METRICS[metric]

    def train(self, X, y): self.X, self.y = X.type(self.dtype), y

    def predict(self, X, batch_size=1, progress=False):
        n_pred = X.shape[0]
        y = torch.zeros(n_pred, dtype=self.y.dtype)
        for start in tqdm(range(0, n_pred, batch_size), desc="Predicting", disable=not progress):
            end = min(start+batch_size, n_pred)
            y[start:end] = self.y[self.fn(self.X[None, :, :] - X[start:end, None, :]).sum(dim=2).argmin(dim=1)]

        return y

In [6]:
n_test = 1000

In [8]:
nn = NearestNeighbor(metric="l2")
nn.train(X_tr, Y_tr)
Y_pred = nn.predict(X_te[:n_test], batch_size=1, progress=True)
acc = torch.mean(Y_pred == Y_te[:n_test], dtype=torch.float16)
print(f"acc = {acc:.4f}")

Predicting: 100%|██████████| 1000/1000 [00:19<00:00, 50.02it/s]

acc = 0.3530


In [5]:
n = NearestNeighbor(metric="l2")
nn.train(X_tr, Y_tr)
Y_pred = nn.predict(X_te, progress=True)
acc = torch.mean(Y_pred == Y_te, dtype=torch.float16)
print(f"acc = {acc:.4f}")

Predicting: 100%|██████████| 10000/10000 [03:22<00:00, 49.36it/s]

acc = 0.3540
